In [1]:
import os
import numpy as np
import pandas as pd

# Define all standard missing value representations used by Stats NZ
NA_PATTERNS = [
    "n/a",
    "N/A",
    "NA",
    "null",
    "NULL",
    "None",
    "",
    " ",
    "..",
    "C",
    "S",
]


def run_data_quality_audit(file_path: str, dataset_label: str):
    """Performs a comprehensive data quality check on a target CSV file and displays:

    1. File Overview (Size, Rows, Columns, Duplicates, Memory)
    2. Column-by-Column Hygiene Table (Data Types, Null Counts, Invalid
    Numerics)
    3. Anomaly Summary Alerts
    """
    print("=" * 85)
    print(f"AUDIT REPORT: {dataset_label.upper()}")
    print("=" * 85)

    if not os.path.exists(file_path):
        print(f"❌ File not found at path: {file_path}")
        return None

    # Calculate file size
    file_size_mb = os.path.getsize(file_path) / (1024 * 1024)

    # Read CSV with explicit NA handling
    df = pd.read_csv(
        file_path,
        na_values=NA_PATTERNS,
        keep_default_na=True,
        low_memory=False,
    )

    total_rows = len(df)
    duplicate_rows = df.duplicated().sum()
    duplicate_pct = (duplicate_rows / total_rows) * 100 if total_rows > 0 else 0
    memory_mb = df.memory_usage(deep=True).sum() / (1024 * 1024)

    # Overview Output
    print(f"📁 File Path            : {file_path}")
    print(f"📦 File Size            : {file_size_mb:.2f} MB")
    print(f"📊 Total Records (Rows) : {total_rows:,}")
    print(f"📋 Total Features (Cols): {df.shape[1]}")
    print(
        f"🔁 Duplicate Rows       : {duplicate_rows:,} ({duplicate_pct:.2f}%)"
    )
    print(f"💾 Memory Allocation    : {memory_mb:.2f} MB\n")

    # Column Breakdown
    col_audit = []
    for col in df.columns:
        col_series = df[col]
        null_cnt = col_series.isna().sum()
        null_pct = (null_cnt / total_rows) * 100 if total_rows > 0 else 0
        inferred_type = str(col_series.dtype)
        unique_cnt = col_series.nunique(dropna=True)

        # Check for non-numeric/dirty strings in object columns
        invalid_num_cnt = 0
        if inferred_type == "object":
            converted = pd.to_numeric(col_series, errors="coerce")
            invalid_num_cnt = (col_series.notna() & converted.isna()).sum()

        sample_vals = col_series.dropna().unique()[:3]
        sample_str = (
            ", ".join(map(str, sample_vals)) if len(sample_vals) > 0 else "N/A"
        )

        col_audit.append({
            "Column Name": col,
            "Data Type": inferred_type,
            "Null Count": null_cnt,
            "Null %": round(null_pct, 2),
            "Unique Values": unique_cnt,
            "Dirty Numeric Strings": invalid_num_cnt,
            "Sample Values": sample_str,
        })

    audit_df = pd.DataFrame(col_audit)

    print("-" * 85)
    print("COLUMN-LEVEL DATA HYGIENE SUMMARY")
    print("-" * 85)
    display(audit_df)

    # Anomaly Highlights
    print("\n" + "-" * 85)
    print("ANOMALY HIGHLIGHTS")
    print("-" * 85)
    high_nulls = audit_df[audit_df["Null %"] > 40.0]["Column Name"].tolist()
    if high_nulls:
        print(f"⚠️  High Missing Data Warning (>40% missing): {high_nulls}")
    else:
        print("✅ No columns exceed the 40% missing threshold.")

    if duplicate_rows > 0:
        print(f"⚠️  Duplicate Warning: Found {duplicate_rows:,} duplicate rows.")
    else:
        print("✅ Zero duplicate rows found.")

    return df

In [2]:
# -----------------------------------------------------------------------------
# STEP 1: AUDIT QUARTERLY EMPLOYMENT SURVEY (QES)
# -----------------------------------------------------------------------------
qes_path = r"D:\STATS NZ DATASET\labour-market-statistics-june-2026\qes-jun26qtr-csv.csv"

df_qes = run_data_quality_audit(
    file_path=qes_path,
    dataset_label="Quarterly Employment Survey (QES) - June 2026",
)

AUDIT REPORT: QUARTERLY EMPLOYMENT SURVEY (QES) - JUNE 2026
📁 File Path            : D:\STATS NZ DATASET\labour-market-statistics-june-2026\qes-jun26qtr-csv.csv
📦 File Size            : 37.91 MB
📊 Total Records (Rows) : 200,156
📋 Total Features (Cols): 13
🔁 Duplicate Rows       : 0 (0.00%)
💾 Memory Allocation    : 131.62 MB

-------------------------------------------------------------------------------------
COLUMN-LEVEL DATA HYGIENE SUMMARY
-------------------------------------------------------------------------------------


,Column Name,Data Type,Null Count,Null %,Unique Values,Dirty Numeric Strings,Sample Values
0,Series_reference,object,0,0.00,1363,200156,"QEMQ.SAAB1A, QEMQ.SAAB1B, QEMQ.SAAB1Z"
1,Period,float64,0,0.00,150,0,"1989.03, 1989.06, 1989.09"
2,Data_value,float64,2142,1.07,53661,0,"14.16, 14.26, 14.24"
3,STATUS,object,0,0.00,2,200156,"FINAL, REVISED"
4,UNITS,object,0,0.00,3,200156,"Dollars, Percent, Number"
5,MAGNTUDE,int64,0,0.00,1,0,0
6,Subject,object,0,0.00,1,200156,Quarterly Employment Survey - QEM
7,Group,object,0,0.00,36,200156,Average Hourly Earnings by Industry (ANZSIC06)...
8,Series_title_1,object,0,0.00,25,200156,"Forestry and Mining, Manufacturing, Electricit..."
9,Series_title_2,object,5100,2.55,5,195056,"Male, Female, Total Both Sexes"



-------------------------------------------------------------------------------------
ANOMALY HIGHLIGHTS
-------------------------------------------------------------------------------------
⚠️  High Missing Data Warning (>40% missing): ['Series_title_4', 'Series_title_5']
✅ Zero duplicate rows found.


In [3]:
# -----------------------------------------------------------------------------
# STEP 2: AUDIT LABOUR MARKET STATISTICS TABLES (LMS)
# -----------------------------------------------------------------------------
lms_path = r"D:\STATS NZ DATASET\labour-market-statistics-june-2026\lms-jun26qtr-tables.csv"

df_lms = run_data_quality_audit(
    file_path=lms_path,
    dataset_label="Labour Market Statistics Tables (LMS) - June 2026",
)

AUDIT REPORT: LABOUR MARKET STATISTICS TABLES (LMS) - JUNE 2026
📁 File Path            : D:\STATS NZ DATASET\labour-market-statistics-june-2026\lms-jun26qtr-tables.csv
📦 File Size            : 273.19 MB
📊 Total Records (Rows) : 1,511,151
📋 Total Features (Cols): 13
🔁 Duplicate Rows       : 0 (0.00%)
💾 Memory Allocation    : 1026.51 MB

-------------------------------------------------------------------------------------
COLUMN-LEVEL DATA HYGIENE SUMMARY
-------------------------------------------------------------------------------------


,Column Name,Data Type,Null Count,Null %,Unique Values,Dirty Numeric Strings,Sample Values
0,Series_reference,object,0,0.00,28985,1511151,"HLFQ.S1A1S, HLFQ.S1A2S, HLFQ.S1A3S"
1,Period,float64,0,0.00,162,0,"1986.03, 1986.06, 1986.09"
2,Data_value,float64,72483,4.80,71948,0,"950.0, 948.0, 943.0"
3,STATUS,object,0,0.00,3,1511151,"REVISED, FINAL, CONFIDENTIAL"
4,UNITS,object,0,0.00,6,1511151,"Number, Percent, number"
5,MAGNTUDE,int64,0,0.00,2,0,"3, 0"
6,Subject,object,0,0.00,3,1511151,"Household Labour Force Survey - HLF, Quarterly..."
7,Group,object,0,0.00,140,1511151,Labour Force Status by Sex: Seasonally Adjuste...
8,Series_title_1,object,0,0.00,133,1511151,"Persons Employed in Labour Force, Persons Unem..."
9,Series_title_2,object,11396,0.75,227,1499755,"Male, Female, Total Both Sexes"



-------------------------------------------------------------------------------------
ANOMALY HIGHLIGHTS
-------------------------------------------------------------------------------------
⚠️  High Missing Data Warning (>40% missing): ['Series_title_4', 'Series_title_5']
✅ Zero duplicate rows found.


In [4]:
# -----------------------------------------------------------------------------
# STEP 3: AUDIT LABOUR COST INDEX (LCI)
# -----------------------------------------------------------------------------
lci_path = r"D:\STATS NZ DATASET\labour-market-statistics-june-2026\lci-jun26qtr-csv.csv"

df_lci = run_data_quality_audit(
    file_path=lci_path,
    dataset_label="Labour Cost Index (LCI) - June 2026",
)

AUDIT REPORT: LABOUR COST INDEX (LCI) - JUNE 2026
📁 File Path            : D:\STATS NZ DATASET\labour-market-statistics-june-2026\lci-jun26qtr-csv.csv
📦 File Size            : 6.42 MB
📊 Total Records (Rows) : 32,938
📋 Total Features (Cols): 13
🔁 Duplicate Rows       : 0 (0.00%)
💾 Memory Allocation    : 22.22 MB

-------------------------------------------------------------------------------------
COLUMN-LEVEL DATA HYGIENE SUMMARY
-------------------------------------------------------------------------------------


,Column Name,Data Type,Null Count,Null %,Unique Values,Dirty Numeric Strings,Sample Values
0,Series_reference,object,0,0.00,398,32938,"LCIQ.SD53Z9, LCIQ.SG11O1, LCIQ.SG11Z9"
1,Period,float64,0,0.00,149,0,"1992.12, 1993.03, 1993.06"
2,Data_value,float64,12,0.04,1943,0,"610.484966, 612.316421, 614.147876"
3,STATUS,object,0,0.00,2,32938,"FINAL, REVISED"
4,UNITS,object,0,0.00,3,32938,"index, Index, Percent"
5,MAGNTUDE,int64,0,0.00,1,0,0
6,Subject,object,0,0.00,1,32938,Labour Cost Index - LCI
7,Group,object,0,0.00,22,32938,"All Sectors Combined, All Salary and Wage Rate..."
8,Series_title_1,object,0,0.00,8,32938,"All Sectors Combined, Salary and Ordinary Time..."
9,Series_title_2,object,0,0.00,68,32938,"All Salary and Wage Rates, Local Government Ad..."



-------------------------------------------------------------------------------------
ANOMALY HIGHLIGHTS
-------------------------------------------------------------------------------------
⚠️  High Missing Data Warning (>40% missing): ['Series_title_3', 'Series_title_4', 'Series_title_5']
✅ Zero duplicate rows found.


In [5]:
# -----------------------------------------------------------------------------
# STEP 4: AUDIT HOUSEHOLD LABOUR FORCE SURVEY (HLFS)
# -----------------------------------------------------------------------------
hlfs_path = r"D:\STATS NZ DATASET\labour-market-statistics-june-2026\hlfs-jun26qtr-csv.csv"

df_hlfs = run_data_quality_audit(
    file_path=hlfs_path,
    dataset_label="Household Labour Force Survey (HLFS) - June 2026",
)

AUDIT REPORT: HOUSEHOLD LABOUR FORCE SURVEY (HLFS) - JUNE 2026
📁 File Path            : D:\STATS NZ DATASET\labour-market-statistics-june-2026\hlfs-jun26qtr-csv.csv
📦 File Size            : 403.45 MB
📊 Total Records (Rows) : 1,278,057
📋 Total Features (Cols): 58
🔁 Duplicate Rows       : 0 (0.00%)
💾 Memory Allocation    : 2593.98 MB

-------------------------------------------------------------------------------------
COLUMN-LEVEL DATA HYGIENE SUMMARY
-------------------------------------------------------------------------------------


,Column Name,Data Type,Null Count,Null %,Unique Values,Dirty Numeric Strings,Sample Values
0,STATUS,object,0,0.00,3,1278057,"REVISED, FINAL, CONFIDENTIAL"
1,SER_NBR,int64,0,0.00,27224,0,"9269, 9270, 9271"
2,Series_reference,object,0,0.00,27224,1278057,"HLFQ.S1A1S, HLFQ.S1A2S, HLFQ.S1A3S"
3,Period,float64,0,0.00,162,0,"1986.03, 1986.06, 1986.09"
4,Data_value,float64,70329,5.50,20729,0,"950.0, 948.0, 943.0"
5,UNITS,object,0,0.00,3,1278057,"Number, Percent, number"
6,MAGNTUDE,int64,0,0.00,2,0,"3, 0"
7,Subject,object,0,0.00,1,1278057,Household Labour Force Survey - HLF
8,Group,object,0,0.00,82,1278057,Labour Force Status by Sex: Seasonally Adjuste...
9,Age Group 3 brackets,object,1257129,98.36,4,20928,"Aged 15-24 Years, Aged 25-54 Years, Aged 55 Ye..."



-------------------------------------------------------------------------------------
ANOMALY HIGHLIGHTS
-------------------------------------------------------------------------------------
⚠️  High Missing Data Warning (>40% missing): ['Age Group 3 brackets', 'Age Group', 'Age Group 6 brackets', 'Disability age breakdown classification', 'Disability status classification', 'Duration of unemployment', 'Employed and Unemployed Persons, Full-Time and Part-Time Status', 'Employment relationship', 'Employment Status', 'Ethnic Single / Combination', 'Ethnic Total Response', 'Formal study status', 'Highest qualification', 'Hours Worked', 'Household Composition', 'Household Labour Force Status', 'Industry ANZSIC06', 'Industry ANZSIC06 Supplementary', 'Job', 'Job tenure', 'Labour force and education status', 'Labour Force Status', 'Labour force/underutilisation classification', 'Main activity', 'Main Job', 'Methods of seeking employment', 'Occupation ANZSCO Level 1', 'Percentage change from 